# Evolución de la contratación en la Comunidad de Madrid

### Análisis territorial, sectorial y por modalidad contractual (2019–2025)

---

**Fuente de datos:** Servicio Público de Empleo Estatal (SEPE)  
**Ámbito geográfico:** Comunidad de Madrid  
**Periodo de análisis:** 2019–2025  
**Herramientas:** Python · Pandas · NumPy · Matplotlib · Seaborn

---

# 01. Introducción

La contratación laboral es uno de los indicadores que permiten
analizar la evolución del mercado de trabajo.

Este proyecto tiene como finalidad estudiar cómo ha evolucionado
la contratación en la Comunidad de Madrid entre 2019 y 2025,
prestando especial atención a las diferencias territoriales,
sectoriales y por modalidad contractual.

El análisis parte de los datos públicos publicados por el SEPE
y combina técnicas de limpieza, transformación y análisis
exploratorio de datos.

# 2. Objetivos del análisis

El análisis se estructura en torno a las siguientes preguntas:

1. ¿Cómo ha evolucionado el volumen de contratos registrados
   entre 2019 y 2025?

2. ¿Qué diferencias existen entre la contratación indefinida
   y temporal?

3. ¿Cómo se distribuye la contratación por sexo?

4. ¿Qué sectores económicos concentran un mayor volumen
   de contratos?

5. ¿Qué diferencias encontramos entre los municipios
   de la Comunidad de Madrid?

6. ¿Qué cambios se observan en la contratación antes y después
   de la reforma laboral de 2021-2022?

El objetivo es identificar tendencias y patrones en los datos,
sin atribuir relaciones causales que el análisis descriptivo
no permita demostrar.

# 03. Fuente de datos

Los datos utilizados en este proyecto proceden del Servicio
Público de Empleo Estatal (SEPE), a través de su portal
de datos abiertos.

Se han recopilado siete archivos CSV, correspondientes
a los años 2019 a 2025.

### Variables disponibles

Los archivos contienen información sobre:

- Periodo de referencia: mes y año.
- Ámbito territorial: comunidad autónoma, provincia y municipio.
- Volumen total de contratos registrados.
- Contratación por sexo y modalidad contractual.
- Contratación por sector económico.

### Consideraciones metodológicas

Antes de realizar comparaciones temporales, se comprobará
la homogeneidad de las variables y las posibles diferencias
en la estructura de los datos.

También se revisarán los valores suprimidos por motivos
de confidencialidad y la integridad de los periodos disponibles.

# 4. Librerías y configuración

En esta sección importamos las librerías que utilizaremos
durante el proyecto.

| Librería | Utilidad |
|---|---|
| Pandas | Manipulación y análisis de datos |
| NumPy | Operaciones numéricas |
| Matplotlib | Creación de gráficos |
| Seaborn | Visualización estadística |
| Pathlib | Gestión de rutas y archivos |

In [1]:
# Librerías para el análisis y tratamiento de datos

import pandas as pd
import numpy as np

# Librerías para visualización

import matplotlib.pyplot as plt
import seaborn as sns

# Gestión de archivos y directorios

from pathlib import Path

# Configuración visual de los gráficos

sns.set_theme(style="whitegrid", palette="deep")

# Mostrar todas las columnas de los DataFrames

pd.set_option("display.max_columns", None)

# 5. Carga de datos

En esta fase realizamos la lectura de los archivos originales publicados por el SEPE.

Disponemos de siete archivos CSV, correspondientes al periodo comprendido entre 2019 y 2025. Cada archivo contiene información mensual sobre los contratos registrados en los municipios españoles.

Los archivos incluyen información territorial, volumen de contratación, modalidad contractual, sexo y sector de actividad económica.

Para facilitar el análisis posterior, almacenaremos inicialmente cada año en un diccionario de DataFrames. De esta forma podremos consultar y comparar los datos de cada ejercicio de manera independiente.

Consideraciones durante la importación:

Los archivos utilizan el punto y coma (;) como separador.

Se omite la primera fila, que contiene el título del informe.

Se utiliza la codificación cp1252 para conservar correctamente los caracteres especiales.

Las columnas se importan inicialmente como texto para evitar conversiones incorrectas, especialmente en los valores suprimidos por confidencialidad.

In [2]:
# Identificamos la carpeta del proyecto
ruta_proyecto = Path.cwd()

# Comprobamos si estamos trabajando desde la carpeta notebooks
if not (ruta_proyecto / "data" / "raw").exists():
    ruta_proyecto = ruta_proyecto.parent

# Definimos la ruta de los archivos originales
ruta_datos = ruta_proyecto / "data" / "raw"

# Comprobamos que la carpeta existe
print(f"Carpeta de datos: {ruta_datos}")
print(f"¿Existe la carpeta?: {ruta_datos.exists()}")

Carpeta de datos: c:\Users\Carmen\Documents\Proyectos Data\SEPE_Contratación_Madrid_2019_2025\data\raw
¿Existe la carpeta?: True


## 5.2. Comprobación de los archivos disponibles

In [3]:
# Listamos los archivos CSV disponibles
archivos_csv = sorted(ruta_datos.glob("*.csv"))

print(f"Archivos encontrados: {len(archivos_csv)}\n")

for archivo in archivos_csv:
    print(archivo.name)

Archivos encontrados: 7

Contratos_por_municipios_2019_csv.csv
Contratos_por_municipios_2020_csv.csv
Contratos_por_municipios_2021_csv.csv
Contratos_por_municipios_2022_csv.csv
Contratos_por_municipios_2023_csv.csv
Contratos_por_municipios_2024_csv.csv
Contratos_por_municipios_2025_csv.csv


## 5.3. Importación de los datos

In [4]:
# Diccionario para almacenar los DataFrames
datos_anuales = {}

# Importación de los archivos correspondientes a 2019-2025
for anio in range(2019, 2026):

    # Localizamos el archivo del año correspondiente
    coincidencias = list(
        ruta_datos.glob(f"Contratos_por_municipios_{anio}_csv*.csv")
    )

    if len(coincidencias) != 1:
        raise ValueError(
            f"Para {anio} se esperaba un archivo, "
            f"pero se encontraron {len(coincidencias)}."
        )

    archivo = coincidencias[0]

    # Leemos el archivo CSV
    df = pd.read_csv(
        archivo,
        sep=";",
        encoding="cp1252",
        skiprows=1,
        dtype=str
    )

    # Añadimos el año de referencia
    df["Anio"] = anio

    # Guardamos el DataFrame
    datos_anuales[anio] = df

    print(
        f"{anio}: {df.shape[0]:,} registros | "
        f"{df.shape[1]} columnas"
    )

2019: 97,524 registros | 20 columnas
2020: 89,446 registros | 20 columnas
2021: 97,608 registros | 20 columnas
2022: 97,608 registros | 20 columnas
2023: 97,608 registros | 20 columnas
2024: 97,608 registros | 20 columnas
2025: 97,620 registros | 20 columnas


# 6. Comprobación de la carga
Una vez importados los archivos, realizamos una primera comprobación de su estructura.

El objetivo es verificar que todos los ejercicios se han cargado correctamente, que mantienen una estructura homogénea y que contienen las variables necesarias para el análisis posterior.

In [5]:
# Resumen de dimensiones de los DataFrames
resumen_carga = pd.DataFrame({
    "Año": datos_anuales.keys(),
    "Filas": [df.shape[0] for df in datos_anuales.values()],
    "Columnas": [df.shape[1] for df in datos_anuales.values()]
})

resumen_carga

,Año,Filas,Columnas
0,2019,97524,20
1,2020,89446,20
2,2021,97608,20
3,2022,97608,20
4,2023,97608,20
5,2024,97608,20
6,2025,97620,20


In [6]:
# Comprobamos la estructura de columnas de cada año
columnas_2025 = set(datos_anuales[2025].columns)

for anio, df in datos_anuales.items():

    mismas_columnas = set(df.columns) == columnas_2025

    print(f"{anio}: mismas columnas que 2025 -> {mismas_columnas}")

2019: mismas columnas que 2025 -> True
2020: mismas columnas que 2025 -> True
2021: mismas columnas que 2025 -> True
2022: mismas columnas que 2025 -> True
2023: mismas columnas que 2025 -> True
2024: mismas columnas que 2025 -> True
2025: mismas columnas que 2025 -> True


# 7. Filtrado geográfico: Comunidad de Madrid

In [11]:
# Filtramos los registros correspondientes a Madrid

datos_madrid = {}

for anio, df in datos_anuales.items():

    df_madrid = df[
        df["Comunidad Autónoma"].str.strip().str.upper()
        == "MADRID, COMUNIDAD DE"
    ].copy()

    datos_madrid[anio] = df_madrid

    print(
        f"{anio}: {len(df_madrid):,} registros de Madrid"
    )

2019: 2,148 registros de Madrid
2020: 1,969 registros de Madrid
2021: 2,148 registros de Madrid
2022: 2,148 registros de Madrid
2023: 2,148 registros de Madrid
2024: 2,148 registros de Madrid
2025: 2,148 registros de Madrid


In [12]:
# Comprobamos los municipios y meses disponibles por año

for anio, df in datos_madrid.items():

    municipios = df[" Municipio"].nunique()
    meses = df["mes"].nunique()

    print(
        f"{anio}: {municipios} municipios | "
        f"{meses} meses | {len(df):,} registros"
    )

2019: 179 municipios | 12 meses | 2,148 registros
2020: 179 municipios | 11 meses | 1,969 registros
2021: 179 municipios | 12 meses | 2,148 registros
2022: 179 municipios | 12 meses | 2,148 registros
2023: 179 municipios | 12 meses | 2,148 registros
2024: 179 municipios | 12 meses | 2,148 registros
2025: 179 municipios | 12 meses | 2,148 registros


## 7.1. Resultados del filtrado territorial

Se ha aplicado un filtro sobre la variable Comunidad Autónoma, seleccionando exclusivamente los registros correspondientes a la Comunidad de Madrid.

Tras el filtrado, se han identificado 179 municipios en todos los ejercicios analizados.

La mayoría de los años presentan 12 meses de información, con 2.148 registros por ejercicio (179 municipios × 12 meses).

Sin embargo, el año 2020 presenta únicamente 11 meses disponibles, con un total de 1.969 registros.

Consideración metodológica: esta diferencia de cobertura temporal deberá tenerse en cuenta en los análisis comparativos. No se tratará 2020 como un ejercicio completo de 12 meses.

# 08. Análisis de calidad de los datos

Antes de iniciar la fase de transformación y análisis, es necesario evaluar la calidad de los datos disponibles.

En esta sección revisaremos la integridad, consistencia y estructura de los registros correspondientes a la Comunidad de Madrid.

El objetivo es identificar posibles problemas que puedan afectar a los resultados, documentar las incidencias detectadas y establecer los criterios de tratamiento que aplicaremos posteriormente.